# ITM-390: ASD Screening in Children - Model Development & Tuning
**Group 10** · American University of Phnom Penh · Advisor: Kuntha PIN  
*Comparative Analysis of Machine Learning Models for Autism Spectrum Disorder Screening in Children*

### Notebook Objectives:
1. Construct leak-free scikit-learn `Pipeline` with `ColumnTransformer`.
2. Establish **Repeated Stratified 5-Fold Cross-Validation (5 repeats = 25 folds)** on training data.
3. Tune **Logistic Regression** (regularization $C$, penalty L1/L2).
4. Sweep **K-Nearest Neighbors** ($K = 1$ to $25$, distance metrics) for bias-variance diagnostic.
5. Tune **Random Forest** (depth, trees, class weights).
6. Construct **Soft Voting Ensemble** (Creative Idea 3.1).
7. Compare CV results across Feature Sets A, B, and C.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocessing import clean_raw_data, load_raw_data, get_feature_sets, train_test_split_dataset, TARGET_COL
from src.models import tune_classifier, sweep_knn_k, build_voting_ensemble, get_repeated_stratified_kfold
from src.evaluate import evaluate_cv_pipeline
from src.utils import set_seed, format_markdown_table

set_seed(42)
print('Environment and modules loaded.')

## 1. Load Data & Prepare Stratified Splits

In [ ]:
df_raw = load_raw_data(project_root / 'data' / 'raw' / 'Autism-Child-Data.arff')
df = clean_raw_data(df_raw, drop_leakage=True)
train_df, test_df = train_test_split_dataset(df, test_size=0.2, random_state=42)

feature_sets = get_feature_sets()
primary_cols = feature_sets['Set A']
y_train = train_df[TARGET_COL].values
cv = get_repeated_stratified_kfold(n_splits=5, n_repeats=5, random_state=42)
print(f'Training instances: {len(train_df)} | Validation folds: 25 folds (Repeated Stratified 5-Fold)')

## 2. K-Nearest Neighbors: K-Sweep Diagnostic (Bias-Variance Tradeoff)

In [ ]:
k_sweep = sweep_knn_k(train_df, y_train, primary_cols, cv=cv)

plt.figure(figsize=(8, 4.5))
plt.errorbar(k_sweep['k_values'], k_sweep['mean_recall'], yerr=k_sweep['std_recall'], fmt='-o', capsize=4, label='Recall (Sensitivity)')
plt.errorbar(k_sweep['k_values'], k_sweep['mean_accuracy'], yerr=k_sweep['std_accuracy'], fmt='-s', capsize=4, label='Accuracy')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Cross-Validation Score (Mean ± Std)')
plt.title('KNN Bias-Variance Tradeoff: K vs Screening Performance')
plt.xticks(k_sweep['k_values'])
plt.legend()
plt.tight_layout()
plt.show()

## 3. Hyperparameter Tuning (Scoring = Recall)

In [ ]:
tuned_models = {}
for model_name in ['Logistic Regression', 'K-Nearest Neighbors', 'Random Forest']:
    pipe, params, score = tune_classifier(model_name, train_df, y_train, primary_cols, cv=cv, scoring='recall')
    tuned_models[model_name] = pipe
    print(f'[{model_name}] Best CV Recall: {score:.4f} | Best Params: {params}')

# Build Soft Voting Ensemble
ensemble = build_voting_ensemble(tuned_models, voting='soft')
ensemble.fit(train_df[primary_cols], y_train)
tuned_models['Voting Ensemble'] = ensemble
print('Voting Ensemble constructed successfully!')

## 4. Cross-Validation Comparison Table

In [ ]:
records = []
for name, pipe in tuned_models.items():
    res = evaluate_cv_pipeline(pipe, train_df, y_train, primary_cols, cv)
    records.append({
        'Model': name,
        'CV Recall': f"{res['recall']['mean']:.4f} ± {res['recall']['std']:.4f}",
        'CV Accuracy': f"{res['accuracy']['mean']:.4f} ± {res['accuracy']['std']:.4f}",
        'CV ROC-AUC': f"{res['roc_auc']['mean']:.4f} ± {res['roc_auc']['std']:.4f}",
        'CV F1': f"{res['f1']['mean']:.4f} ± {res['f1']['std']:.4f}"
    })
cv_df = pd.DataFrame(records)
print(format_markdown_table(cv_df))

## 5. Feature Set Impact Analysis (Set A vs Set B vs Set C)

In [ ]:
feat_comp_csv = project_root / 'results' / 'metrics' / 'feature_sets_comparison.csv'
if feat_comp_csv.exists():
    feat_df = pd.read_csv(feat_comp_csv)
    display(feat_df)
else:
    print('Run main.py to generate full feature sets comparison table.')

> **Key Finding:** Models trained on Set C (Demographics only) achieve only ~53-55% recall, confirming that the behavioral questions `A1`..`A10` drive the predictive power of the screening system.